# Assignment 1 – Cognitive Decline Activity Recognition
## COEN 498 / COEN 691 – Wearable Computing

**Objective:** Train and evaluate activity recognition models on wrist-worn smartwatch accelerometer data as a proxy biomarker for cognitive decline.

**Dataset:** `df_train.csv` — ~1.48 M rows at 10 Hz with columns:
`stamp | user_acceleration_x | user_acceleration_y | user_acceleration_z | user_activity_label`

**Activities:** Work · Other · Eat · Travel · Hygiene · Cook · Exercise

---
> ⚠️ **Run all cells top-to-bottom.** Paste your `src/` code into the appropriate code cells below.

## Section 1 – Setup & Imports

Configure CPU threading **before** importing numpy/scipy/sklearn (same pattern as `tutorial_3_optimized.py`).  
Detect GPU for LSTM; fall back to maximally-threaded CPU.

In [ ]:
# ============================================================
# CRITICAL: Set BLAS/OpenMP threads BEFORE any numpy/sklearn imports
# ============================================================
import os
import multiprocessing

_num_cores = str(multiprocessing.cpu_count())
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'BLIS_NUM_THREADS'):
    os.environ.setdefault(_v, _num_cores)

print(f"[INIT] BLAS threads → {_num_cores} cores")

# ============================================================
# Standard imports (after env vars)
# ============================================================
import sys
import pathlib
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from scipy.signal import butter, filtfilt, medfilt
from scipy.stats import mode
from numpy.lib.stride_tricks import as_strided as ast

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ---- Device selection ----
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"[INIT] PyTorch device: CUDA ({torch.cuda.get_device_name(0)})")
else:
    n = int(_num_cores)
    torch.set_num_threads(n)
    torch.set_num_interop_threads(max(1, n // 2))
    DEVICE = torch.device('cpu')
    print(f"[INIT] PyTorch device: CPU (torch threads={n})")

# ---- Reproducibility ----
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"[INIT] NumPy seed={SEED}, PyTorch seed set")

# ============================================================
# Paths (adjust if running in Colab)
# ============================================================
if os.path.exists('/content'):                          # Google Colab
    CSV_PATH    = '/content/df_train.csv'
    RESULTS_DIR = pathlib.Path('/content/results')
    IMAGES_DIR  = pathlib.Path('/content/images')
else:                                                   # Local
    _BASE       = pathlib.Path('.')                     # run from assignment_1/
    CSV_PATH    = str(_BASE / 'df_train.csv')
    RESULTS_DIR = _BASE / 'results'
    IMAGES_DIR  = _BASE / 'images'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print(f"[INIT] CSV path  : {CSV_PATH}")
print(f"[INIT] Results   : {RESULTS_DIR}")
print(f"[INIT] Images    : {IMAGES_DIR}")

## Section 2 – Data Loading & Verification

Load `df_train.csv`, inspect schema and class distribution.  
Paste code from `src/data.py → load_data()` here.

In [ ]:
COL_TIME  = 'stamp'
COL_X     = 'user_acceleration_x'
COL_Y     = 'user_acceleration_y'
COL_Z     = 'user_acceleration_z'
COL_LABEL = 'user_activity_label'

print(f"[DATA] Loading from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH, parse_dates=[COL_TIME])

print(f"[DATA] Shape        : {df.shape}")
print(f"[DATA] Columns      : {df.columns.tolist()}")
print(f"[DATA] Missing vals : {df.isnull().sum().sum()}")
print(f"\n[DATA] Class distribution:\n{df[COL_LABEL].value_counts().to_string()}")

# Assertions
assert COL_LABEL in df.columns, "Missing label column!"
assert df.isnull().sum().sum() == 0, "Dataset has NaN values – handle before proceeding"

display(df.head())
display(df.dtypes.rename('dtype'))

## Section 3 – Data Preprocessing & Filtering

- Add Magnitude channel  
- Sort chronologically (critical to prevent data leakage)  
- Optional Butterworth / median filter (paste config from `src/data.py → apply_filter()`)

In [ ]:
SAMPLING_FREQ = 10       # Hz (inferred from 0.1 s timestamp increments)

# Paste src/data.py → apply_filter() here if you want filtering
# Example (no filter by default):
FILTER_CONFIG = None
# FILTER_CONFIG = {'method': 'lowpass', 'cutoff': 4, 'order': 4}

# --- Sort chronologically ---
df = df.sort_values(COL_TIME).reset_index(drop=True)
print(f"[PREP] Sorted chronologically. Rows: {len(df):,}")

# --- Add Magnitude ---
df['Magnitude'] = np.sqrt(df[COL_X]**2 + df[COL_Y]**2 + df[COL_Z]**2)
print(f"[PREP] Magnitude column added. Columns: {df.columns.tolist()}")

# --- Optional filter (paste apply_filter from src/data.py) ---
if FILTER_CONFIG is not None:
    # TODO: paste apply_filter() definition here and call:
    # df = apply_filter(df, fs=SAMPLING_FREQ, config=FILTER_CONFIG)
    print(f"[PREP] Filter applied: {FILTER_CONFIG}")
else:
    print("[PREP] No filter applied (FILTER_CONFIG=None)")

display(df[[COL_X, COL_Y, COL_Z, 'Magnitude']].describe())

## Section 4 – Feature Extraction (Windowing & Fourier Analysis)

Paste the sliding window + feature extraction helpers from `src/data.py`.  
Output: feature matrix `X_feat` (N_windows × n_features) and raw windows `X_seq` (N_windows × ws × C) for LSTM.

In [ ]:
# ---- Hyperparameters ----
WINDOW_SIZE = 50      # samples (50 × 0.1 s = 5 s windows)
STEP_SIZE   = 25      # 50 % overlap
CHANNELS    = [COL_X, COL_Y, COL_Z, 'Magnitude']

# ---- Paste functions from src/data.py: ----
#   _norm_shape(), sliding_window(), perform_sliding_window()
#   _window_features(), extract_features(), encode_labels()
# TODO ↓

# After pasting, run:
#
# le, y_all = encode_labels(df[COL_LABEL])
# x_raw = df[CHANNELS].values
# X_seq, y_win = perform_sliding_window(x_raw, y_all, WINDOW_SIZE, STEP_SIZE)
# X_feat = extract_features(X_seq, verbose=True)

# ---- Placeholder so later cells don't error ----
print("[FEAT] Paste windowing + feature functions from src/data.py, then re-run.")

# Once the above is done, expected output:
# X_feat : shape (N_windows, 24)   [4 channels × 6 stats]
# X_seq  : shape (N_windows, 50, 4)
# y_win  : shape (N_windows,)

## Section 5 – Exploratory Data Analysis (EDA)

All plots go to `images/`.  
Paste/call from `src/plot.py`: `plot_activity_distribution`, `plot_acceleration_samples`, `plot_frequency_content`, `plot_temporal_patterns`, `plot_correlation_heatmap`.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # remove if interactive display is available

_PALETTE = 'tab10'
_images_dir = str(IMAGES_DIR)

# Paste plot functions from src/plot.py here or call the module:
# from src.plot import run_eda_plots
# run_eda_plots(df.sample(100_000, random_state=42), images_dir=_images_dir)

# ---- 5.1 Activity Distribution ----
fig, ax = plt.subplots(figsize=(9, 4))
counts = df[COL_LABEL].value_counts().sort_values(ascending=False)
ax.bar(counts.index, counts.values,
       color=sns.color_palette(_PALETTE, len(counts)))
ax.set_title('Activity Class Distribution', fontweight='bold')
ax.set_ylabel('Sample Count')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
fig.savefig(f'{_images_dir}/activity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("[EDA] activity_distribution.png saved")

# ---- 5.2 Acceleration samples per activity ----
labels = df[COL_LABEL].unique()
n_samples = SAMPLING_FREQ * 5  # 5 seconds
fig, axes = plt.subplots(len(labels), 1, figsize=(12, 3 * len(labels)))
if len(labels) == 1: axes = [axes]
palette = sns.color_palette(_PALETTE, len(labels))
for ax, lbl, col in zip(axes, labels, palette):
    sub = df[df[COL_LABEL] == lbl][[COL_X, COL_Y, COL_Z]].head(n_samples)
    t = np.arange(len(sub)) / SAMPLING_FREQ
    ax.plot(t, sub[COL_X].values, label='X', lw=0.8)
    ax.plot(t, sub[COL_Y].values, label='Y', lw=0.8)
    ax.plot(t, sub[COL_Z].values, label='Z', lw=0.8)
    ax.set_title(lbl, color=col, fontweight='bold', fontsize=10)
    ax.set_ylabel('Accel (g)')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Acceleration Time-Series Samples', fontsize=12, y=1.0)
plt.tight_layout()
fig.savefig(f'{_images_dir}/acceleration_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("[EDA] acceleration_samples.png saved")

# ---- 5.3 Frequency Content (FFT) ----
segment = SAMPLING_FREQ * 5
freqs = np.fft.rfftfreq(segment, d=1.0 / SAMPLING_FREQ)
fig, ax = plt.subplots(figsize=(10, 4))
for lbl, col in zip(labels, palette):
    vals = df[df[COL_LABEL] == lbl][COL_Z].values
    n_seg = len(vals) // segment
    if n_seg == 0: continue
    mat = vals[:n_seg * segment].reshape(n_seg, segment)
    mag = np.abs(np.fft.rfft(mat, axis=1)).mean(axis=0)
    ax.plot(freqs, mag, label=lbl, lw=1.2)
ax.set_title(f'FFT Magnitude Spectrum per Activity (Z-axis)', fontweight='bold')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
fig.savefig(f'{_images_dir}/frequency_content.png', dpi=150, bbox_inches='tight')
plt.show()
print("[EDA] frequency_content.png saved")

# ---- 5.4 Temporal Pattern ----
df_t = df.copy()
df_t['hour'] = df_t[COL_TIME].dt.hour
pivot = df_t.groupby(['hour', COL_LABEL]).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(12, 4))
pivot_pct.plot.area(ax=ax, colormap=_PALETTE, alpha=0.85)
ax.set_title('Activity Proportion by Hour of Day', fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Proportion')
ax.legend(title='Activity', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
fig.savefig(f'{_images_dir}/temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("[EDA] temporal_patterns.png saved")

# ---- 5.5 Correlation Heatmap ----
corr = df[[COL_X, COL_Y, COL_Z]].corr()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
ax.set_title('Accelerometer Channel Correlation', fontweight='bold')
plt.tight_layout()
fig.savefig(f'{_images_dir}/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("[EDA] correlation_heatmap.png saved")

## Section 6 – Train / Test Split

Stratified split to preserve class proportions.  
Verify class balance is maintained in both subsets.

In [ ]:
TEST_FRACTION = 0.20

# After feature extraction in Section 4 you should have:
#   X_feat : (N, n_features)
#   X_seq  : (N, WINDOW_SIZE, C)
#   y_win  : (N,)
# Replace the placeholders below with those variables.

# TODO: replace with actual arrays after Section 4 is filled in
# X_feat = ...
# X_seq  = ...
# y_win  = ...

# Stratified split
idx = np.arange(len(X_feat))    # will work once X_feat is defined
tr_idx, te_idx = train_test_split(idx, test_size=TEST_FRACTION,
                                  stratify=y_win, random_state=SEED)

X_train_feat = X_feat[tr_idx];  X_test_feat = X_feat[te_idx]
X_train_seq  = X_seq[tr_idx];   X_test_seq  = X_seq[te_idx]
y_train      = y_win[tr_idx];   y_test      = y_win[te_idx]

# Standardise features (fit only on train)
scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat  = scaler.transform(X_test_feat)

print(f"[SPLIT] Train: {len(y_train):,}  |  Test: {len(y_test):,}")

# Verify class balance
le_classes = le.classes_    # available if encode_labels() was called in Section 4
for split_name, labels in [('Train', y_train), ('Test', y_test)]:
    vals, cts = np.unique(labels, return_counts=True)
    print(f"\n{split_name} distribution:")
    for v, c in zip(vals, cts):
        pct = 100 * c / len(labels)
        print(f"  {le_classes[v]:10s}: {c:6,}  ({pct:.1f}%)")

## Section 7 – Model Definition

Paste the `LSTMClassifier` from `src/model.py`.  
Also show sklearn baselines (SVM / DT / RF) used for comparison.

In [ ]:
NUM_CLASSES  = len(le_classes)    # 7 activities
LSTM_HIDDEN  = 128
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.3

# ---- Paste LSTMClassifier from src/model.py ----

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=4, hidden_size=128,
                 num_layers=2, num_classes=7, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)


model_lstm = LSTMClassifier(
    input_size  = X_train_seq.shape[2],   # number of channels
    hidden_size = LSTM_HIDDEN,
    num_layers  = LSTM_LAYERS,
    num_classes = NUM_CLASSES,
    dropout     = LSTM_DROPOUT,
).to(DEVICE)

# Parameter count
total_params = sum(p.numel() for p in model_lstm.parameters())
print(f"[MODEL] LSTM total parameters: {total_params:,}")
print(model_lstm)

# ---- sklearn baselines ----
svm_model = SVC(kernel='rbf', random_state=SEED)
dt_model  = DecisionTreeClassifier(random_state=SEED)
rf_model  = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
print("\n[MODEL] sklearn models initialised:"
      f"\n  SVM : {svm_model}"
      f"\n  DT  : {dt_model}"
      f"\n  RF  : {rf_model}")

## Section 8 – Model Training with CPU Multi-threading Optimisation

- **LSTM**: PyTorch CUDA if available, else `torch.set_num_threads(N)` (set at top).  
- **sklearn (RF)**: `n_jobs=-1`.  
- **BLAS / OpenMP**: pinned to all cores at process start (Section 1).

In [ ]:
from tqdm.notebook import tqdm as nb_tqdm

LSTM_EPOCHS = 30
LSTM_BATCH  = 256
LSTM_LR     = 1e-3

# ---- LSTM Training Loop ----
X_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_t = torch.tensor(y_train,     dtype=torch.long)
dataset    = TensorDataset(X_t, y_t)
dataloader = DataLoader(dataset, batch_size=LSTM_BATCH, shuffle=True, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=LSTM_LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_losses = []
model_lstm.train()

for epoch in nb_tqdm(range(LSTM_EPOCHS), desc="LSTM Training"):
    epoch_loss = 0.0
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model_lstm(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(y_batch)
    epoch_loss /= len(dataset)
    train_losses.append(epoch_loss)
    scheduler.step()

print(f"[TRAIN] LSTM done. Final loss: {train_losses[-1]:.4f}")

# ---- sklearn training ----
import time
print("\n[TRAIN] Fitting SVM …")
t0 = time.time(); svm_model.fit(X_train_feat, y_train)
print(f"  done in {time.time()-t0:.1f}s")

print("[TRAIN] Fitting Decision Tree …")
t0 = time.time(); dt_model.fit(X_train_feat, y_train)
print(f"  done in {time.time()-t0:.1f}s")

print("[TRAIN] Fitting Random Forest …")
t0 = time.time(); rf_model.fit(X_train_feat, y_train)
print(f"  done in {time.time()-t0:.1f}s")

## Section 9 – Evaluation & Classification Report

Compute predictions, accuracy, weighted F1, macro F1, and full per-class report.  
Save metrics to `results/metrics.txt`.

In [ ]:
from sklearn.metrics import accuracy_score

# ---- LSTM predictions ----
model_lstm.eval()
all_preds = []
X_te_t = torch.tensor(X_test_seq, dtype=torch.float32)
loader  = DataLoader(TensorDataset(X_te_t), batch_size=512, shuffle=False)
with torch.no_grad():
    for (batch,) in loader:
        logits = model_lstm(batch.to(DEVICE))
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
lstm_pred = np.concatenate(all_preds)

# ---- sklearn predictions ----
svm_pred = svm_model.predict(X_test_feat)
dt_pred  = dt_model.predict(X_test_feat)
rf_pred  = rf_model.predict(X_test_feat)

# ---- Metrics ----
predictions = {'LSTM': lstm_pred, 'SVM': svm_pred, 'DT': dt_pred, 'RF': rf_pred}
class_names  = list(le_classes)

report_lines = ["=" * 70,
                "ASSIGNMENT 1 – EVALUATION RESULTS",
                "=" * 70, ""]

for name, y_pred in predictions.items():
    f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_m = f1_score(y_test, y_pred, average='macro',    zero_division=0)
    acc  = accuracy_score(y_test, y_pred)
    rep  = classification_report(y_test, y_pred,
                                 target_names=class_names, zero_division=0)
    header = f"--- {name} | Weighted F1: {f1_w:.4f} | Macro F1: {f1_m:.4f} | Acc: {acc:.4f} ---"
    print(header)
    print(rep)
    report_lines += [header, rep]

# Save to file
metrics_file = RESULTS_DIR / 'metrics.txt'
metrics_file.write_text("\n".join(report_lines))
print(f"[EVAL] Metrics saved to {metrics_file}")

## Section 10 – Confusion Matrix & Evaluation Plots

Normalised confusion matrix + LSTM training loss curve.  
Saved to `images/`.

In [ ]:
def _plot_cm(y_true, y_pred, name, class_names, images_dir):
    """Plot and save a normalised confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                vmin=0, vmax=1, ax=ax, linewidths=0.4)
    ax.set_title(f'Confusion Matrix – {name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted');  ax.set_ylabel('True')
    plt.xticks(rotation=30, ha='right');  plt.yticks(rotation=0)
    plt.tight_layout()
    out = pathlib.Path(images_dir) / f'confusion_matrix_{name.lower()}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[PLOT] {out}")

for model_name, y_pred in predictions.items():
    _plot_cm(y_test, y_pred, model_name, class_names, str(IMAGES_DIR))

# ---- LSTM training loss curve ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(train_losses) + 1), train_losses,
        marker='o', markersize=3, linewidth=1.5, color='steelblue')
ax.set_title('LSTM Training Loss', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
out = IMAGES_DIR / 'lstm_training_loss.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"[PLOT] {out}")

## Section 11 – Save Results & Artifacts

- Save LSTM weights (`torch.save`)  
- Save RF model (`joblib.dump`)  
- Confirm all output files exist  
- Print final summary table

In [ ]:
import joblib

# ---- Save LSTM weights ----
lstm_weights_path = RESULTS_DIR / 'lstm_model.pt'
torch.save(model_lstm.state_dict(), lstm_weights_path)
print(f"[SAVE] LSTM weights: {lstm_weights_path}")

# ---- Save RF model ----
rf_path = RESULTS_DIR / 'rf_model.joblib'
joblib.dump(rf_model, rf_path)
print(f"[SAVE] RF model    : {rf_path}")

# ---- Verify all expected images exist ----
expected_images = [
    'activity_distribution.png',
    'acceleration_samples.png',
    'frequency_content.png',
    'temporal_patterns.png',
    'correlation_heatmap.png',
    'lstm_training_loss.png',
    'confusion_matrix_lstm.png',
    'confusion_matrix_rf.png',
]
print("\n[CHECK] Image files:")
for fname in expected_images:
    p = IMAGES_DIR / fname
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {p}")

# ---- Final summary table ----
print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"{'Model':8s} | {'Weighted F1':>12s} | {'Accuracy':>10s}")
print("-" * 38)
for name, y_pred in predictions.items():
    f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    acc  = accuracy_score(y_test, y_pred)
    print(f"{name:8s} | {f1_w:12.4f} | {acc:10.4f}")
print("=" * 70)
print(f"\n[DONE] All outputs in:\n  Metrics: {RESULTS_DIR}\n  Images : {IMAGES_DIR}")